In [1]:
# train_model.py
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTETomek
import joblib

print("🔍 Loading data...")
data = pd.read_csv('fetal_health.csv')

# --- IQR Capping (outliers handling) ---
print("🧹 Applying IQR capping...")
iqr_capped = data.copy()
num_cols = data.select_dtypes(include=np.number).columns.drop('fetal_health')

for col in num_cols:
    Q1 = iqr_capped[col].quantile(0.25)
    Q3 = iqr_capped[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    iqr_capped[col] = np.clip(iqr_capped[col], lower, upper)

# --- Split ---
print("SplitOptions Features & target...")
X = iqr_capped.drop(columns=['fetal_health'])
y = iqr_capped['fetal_health']

# --- Balance with SMOTETomek ---
print("⚖️ Balancing classes with SMOTETomek...")
smotetomek = SMOTETomek(sampling_strategy='auto', random_state=42)
X_res, y_res = smotetomek.fit_resample(X, y)

# --- Scale ---
print("📏 Scaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_res)

# --- Train KNN ---
print("🧠 Training KNN model...")
knn = KNeighborsClassifier()
knn.fit(X_scaled, y_res)

# --- Save artifacts ---
joblib.dump(knn, 'fetal_knn_model.pkl')
joblib.dump(scaler, 'fetal_scaler.pkl')
joblib.dump(list(X.columns), 'feature_names.pkl')

# --- Precise feature ranges for sliders (with 3 decimals for tiny features) ---
print("🎯 Calculating precise feature ranges for sliders...")
feature_ranges = {}
for col in num_cols:
    min_v = iqr_capped[col].min()
    max_v = iqr_capped[col].max()
    # Use 3 decimals for small values (<1), 1 dec for medium, int for large
    if max_v < 1.0:
        min_v = round(min_v, 3)
        max_v = round(max_v, 3)
        step = 0.001
    elif max_v < 10.0:
        min_v = round(min_v, 2)
        max_v = round(max_v, 2)
        step = 0.01
    else:
        min_v = int(round(min_v, 0))
        max_v = int(round(max_v, 0))
        step = 1
    feature_ranges[col] = (min_v, max_v, step)
joblib.dump(feature_ranges, 'feature_ranges.pkl')

# --- 5 Real, Diverse Examples from Data (covers all 3 classes) ---
# Selected indices manually for clinical diversity
example_indices = {
    "🟢 Normal (Stable)": 1,      # Class 1 — stable baseline, normal variability
    "🟢 Normal (Active Fetus)": 131,  # Class 1 — high fetal movement
    "🟡 Suspect (Borderline)": 10,   # Class 2 — mild decelerations
    "🔴 Pathological (Decelerations)": 5,    # Class 3 — severe/long decelerations
    "🔴 Pathological (High Risk)": 134,  # Class 3 — abnormal STV + high variance
}
preset_examples = {
    name: data.iloc[idx][:-1].tolist()  # exclude target
    for name, idx in example_indices.items()
}
joblib.dump(preset_examples, 'preset_examples.pkl')

print("✅ Training & saving completed!")
print("- fetal_knn_model.pkl")
print("- fetal_scaler.pkl")
print("- feature_names.pkl")
print("- feature_ranges.pkl")
print("- preset_examples.pkl")

🔍 Loading data...
🧹 Applying IQR capping...
SplitOptions Features & target...
⚖️ Balancing classes with SMOTETomek...
📏 Scaling features...
🧠 Training KNN model...
🎯 Calculating precise feature ranges for sliders...
✅ Training & saving completed!
- fetal_knn_model.pkl
- fetal_scaler.pkl
- feature_names.pkl
- feature_ranges.pkl
- preset_examples.pkl
